# model-save-state-dict — worked example 2: Atomic save via tmp + os.replace

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-save-state-dict`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A crash mid-write can leave a half-written checkpoint. The fix: rank 0 writes to a `.tmp` path, then `os.replace(tmp, final)` — a POSIX atomic rename — so readers ever only see the complete file. All ranks barrier afterward.

## Worked solution

We make the rank-0 save crash-resilient.

1. **Write to tmp.** `tmp = path + '.tmp'`, then `t.save(model.state_dict(), tmp)`. If the process dies here, only the tmp file is partial; the real path is untouched.
2. **Atomic rename.** `os.replace(tmp, path)` swaps the complete tmp into the final path in one indivisible operation. After it, `path` is whole and `tmp` no longer exists.
3. **Barrier.** All ranks call `barrier()` so non-zero ranks do not read before the rename completes.
4. **Guarantees.** Post-call: `path` exists, `path + '.tmp'` does not, and any rank loads the same state.

The demo runs the atomic save, then asserts the final file exists, the tmp is gone, and the loaded weights match.

In [ ]:
import torch as t
import torch.nn as nn
import tempfile, os

t.manual_seed(1)

class MockDist:
    def __init__(self):
        self.barrier_calls = 0
    def barrier(self):
        self.barrier_calls += 1

def atomic_save(rank, dist_mod, model, path):
    if rank == 0:
        tmp = path + '.tmp'
        t.save(model.state_dict(), tmp)
        os.replace(tmp, path)
    dist_mod.barrier()
    return path

model = nn.Linear(3, 3, bias=False)
with t.no_grad():
    model.weight.fill_(2.0)

tmp_dir = tempfile.mkdtemp()
path = os.path.join(tmp_dir, 'model.pt')
dist_mod = MockDist()
atomic_save(0, dist_mod, model, path)
print('final exists:', os.path.exists(path))
print('tmp gone:', not os.path.exists(path + '.tmp'))
print('loaded sum:', t.load(path, weights_only=True)['weight'].sum().item())  # 18.0